* 使用行业内的排序，进行因子测试；与回归版本，以及原始因子值版本进行比较。本部分参考自《QEPM》 p.p 117
* 请在环境变量中设置`DB_URI`指向数据库

# 参数设定
--------------

In [ ]:
import asyncio
import datetime as dt


In [ ]:
import numpy as np
import pyarrow as pa


# 样例因子
--------------------

我们下面分三种方法，分别考查这几种方法在避免行业集中上面的效果：

* 使用原始因子的排序；
* 使用原始因子在行业内的排序；
* 使用原始因子在行业哑变量上回归后得到的残差排序。

## 1. 原始因子排序
--------------------

In [ ]:
import pyarrow.compute as pc


In [ ]:
rng = np.random.default_rng(42)
n_dates = 6


对于原始因子，如果我们不做任何行业上面的处理，发现我们选定的alpha因子`CFO2EV`较大的股票集中于银行和大金融板块。

## 2. 行业内排序因子
----------------------

这里我们使用调整后的申万行业分类作为行业标签：

In [ ]:
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]


使用行业内的排序，则行业分布会比较平均。

## 3. 使用回归将因子行业中性
--------------------------------

还有一种思路，使用线性回归，以行业为哑变量，使用回归后的残差作为因子的替代值，做到行业中性：

In [ ]:
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets


In [ ]:
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})


In [ ]:
from alphamind.portfolio.rankbuilder import rank_build
from alphamind.settlement.simplesettle import simple_settle


我们发现这种方法的效果并不是很好。调整的幅度并不是很大，同时仍然存在着集中于大金融板块的问题。

# 回测结果
------------------

我们使用简单等权重做多前20%支股票，做空后20%的方法，考察三种方法的效果：

In [ ]:
labels = panel["industry"].to_pylist()
group_labels = [
    f"{trade_date.isoformat()}:{label}"
    for trade_date, label in zip(panel["trade_date"].to_pylist(), labels)
]


In [ ]:
groups = np.unique(group_labels, return_inverse=True)[1]
signal = panel["factor"].to_numpy(zero_copy_only=False)


In [ ]:
selected = rank_build(signal, use_rank=2, groups=groups).flatten().astype(float)


In [ ]:
weights = selected / 6.0
settlement = await asyncio.to_thread(
    simple_settle,
    weights,
    panel["dx"].to_numpy(zero_copy_only=False),
    groups=labels,
)


In [ ]:
assert selected.sum() == n_dates * 6
settlement
